In [1]:
from google.colab import drive

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
drive.mount('/content/drive')

In [ ]:
!pip install "pillow<11.0.0"

In [ ]:
!pip install --upgrade pip

In [ ]:
!pip install -U git+https://github.com/huggingface/diffusers.git

In [ ]:
!pip install -U accelerate safetensors transformers huggingface_hub

In [ ]:
from huggingface_hub import login
login(token="token")

In [ ]:
import torch
import gc
from diffusers import Flux2KleinPipeline
from PIL import Image

device = "cuda"
dtype = torch.bfloat16

pipe = Flux2KleinPipeline.from_pretrained(
    "black-forest-labs/FLUX.2-klein-4B",
    torch_dtype=dtype,
    device_map="cpu"
)

# Оптимизируем память
pipe.enable_model_cpu_offload()
pipe.enable_sequential_cpu_offload()

print('Модель загружена')

pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

gc.collect()
torch.cuda.empty_cache()
print('Кэш очищен')



In [ ]:
import random
import json
from pathlib import Path

drive_root = Path("/content/drive/MyDrive/TestProject")
input_dir = drive_root / "input_images"
output_dir = drive_root / "output_images"
json_path = drive_root / "editing_results.json"

In [ ]:
PROMPTS = [
    "Change the colors to warm autumn tones",
    "Make the image black and white",
    "Add a cute cat sitting nearby",
    "Change the lighting to golden hour sunset",
    "Change background colour to green",
    "Add a rainbow in the sky"
]

# Получаем список файлов изображений с помощью Path.glob()
image_files = list(input_dir.glob("*.*"))
# Фильтруем только изображения по расширениям
image_extensions = {'.png', '.jpg', '.jpeg', '.webp'}
image_files = [f for f in image_files if f.suffix.lower() in image_extensions]

print(f"Найдено изображений: {len(image_files)}")
results = []

In [ ]:
input_dir_dataset = "input_images/"
output_dir_dataset = "output_images/"

In [ ]:
from PIL import Image

In [ ]:
for idx, image_path in enumerate(image_files, 1):
    print(f"\n Обработка {idx}/{len(image_files)}: {image_path.name}")

    # Загружаем исходное изображение (используем Path объект напрямую)
    original_image = Image.open(image_path).convert("RGB")

    # Выбираем случайный промпт
    prompt = random.choice(PROMPTS)

    # Генерируем имя для выходного файла
    output_filename = f"edited_{image_path.stem}{image_path.suffix}"
    output_path = output_dir / output_filename

    try:
        print(f"Промпт: {prompt[:50]}...")
        edited_image = pipe(
        prompt=prompt,
        image=original_image,  # Передаем исходное изображение для редактирования
        width=original_image.width,
        height=original_image.height,
        num_inference_steps=4,
        guidance_scale=1.0
    ).images[0]
        edited_image.save(output_path)
        print(f"Сохранено: {output_filename}")

        # Сохраняем результат
        results.append({
            "source_image_id": str(input_dir_dataset + image_path.name),
            "prompt": prompt,
            "edited_image_id": str(output_dir_dataset + output_path.name)
        })


    except Exception as e:
        print(f"Ошибка: {e}")



In [ ]:
#Сохраняем датасет в json
with open(json_path, "w", encoding="utf-8") as f:
           json.dump(results, f, indent=4, ensure_ascii=False)